### Extract Information

In [ ]:
'''
def parse_sem_metadata(metadata_text):
    metadata_dict = {}
    current_section = None

    for line in metadata_text.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith("[") and line.endswith("]"):
            current_section = line.strip("[]")
            metadata_dict[current_section] = {}
        else:
            if "=" in line:
                key, value = line.split("=", 1)
                key, value = key.strip(), value.strip()
                try:
                    if "." in value or "e" in value.lower():
                        value = float(value)
                    else:
                        value = int(value)
                except ValueError:
                    pass
                if current_section:
                    metadata_dict[current_section][key] = value
                else:
                    metadata_dict[key] = value

    return metadata_dict
'''

In [ ]:
'''
img = Image.open(imagePath)
metaInfo = {key : img.tag[key] for key in img.tag_v2}
metadata = parse_sem_metadata(metaInfo[34682][0])

pixelWidthNm = (metadata["Scan"]["PixelWidth"])*(10**9) if metadata["Scan"]["PixelWidth"] else 1
pixelHeightNm = (metadata["Scan"]["PixelHeight"])*(10**9) if metadata["Scan"]["PixelHeight"] else 1
ResolutionX = metadata["Image"]["ResolutionX"] if metadata["Image"] else 1500
ResolutionY = metadata["Image"]["ResolutionY"] if metadata["Image"] else 1024
'''

### Image Pre-Processing

In [1]:
'''
def rotationImage(image,angle=0):
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotatedImage = cv2.warpAffine(image, M, (w, h))
    return rotatedImage

def distortionImage(image, value=0):
    camera_matrix = np.array([[5000, 0, image.shape[1]//2], [0, 5000, image.shape[0]//2], [0, 0, 1]])
    correctedImage = cv2.undistort(image, camera_matrix, np.array([value,0,0,0]))
    return correctedImage
'''

'\ndef rotationImage(image,angle=0):\n    (h, w) = image.shape[:2]\n    center = (w // 2, h // 2)\n    M = cv2.getRotationMatrix2D(center, angle, 1.0)\n    rotatedImage = cv2.warpAffine(image, M, (w, h))\n    return rotatedImage\n\ndef distortionImage(image, value=0):\n    camera_matrix = np.array([[5000, 0, image.shape[1]//2], [0, 5000, image.shape[0]//2], [0, 0, 1]])\n    correctedImage = cv2.undistort(image, camera_matrix, np.array([value,0,0,0]))\n    return correctedImage\n'

### Edge detection

In [ ]:
'''
#original
def edge_detect_lin(leading_edges, trailing_edges, Arc, b, threshold):
    rows, cols = Arc.shape
    leading_edge_profiles = np.zeros((rows, len(leading_edges)))
    trailing_edge_profiles = np.zeros((rows, len(trailing_edges)))
    
    for n in range(len(leading_edges)):
        for m in range(rows):
            # Leading edge detection
            s1 = max(leading_edges[n] - b, 0)
            s2 = min(leading_edges[n] + b, cols - 1)
            segment = Arc[m, s1:s2+1]
            x = np.arange(s1, s2 + 1)
            
            #leading_edge_profiles[m, n] = x[np.argmax(segment)]

            
            if len(segment) > 1:
                A = np.vstack([x, np.ones(len(segment))]).T
                bb = np.linalg.lstsq(A, segment, rcond=None)[0]  # Solve Ax = b
                slope, intercept = bb  # Unpack slope and intercept
                p = (threshold - intercept) / slope  # Compute refined edge position
                leading_edge_profiles[m, n] = p
            
            
            # Trailing edge detection
            s1b = max(trailing_edges[n] - b, 0)
            s2b = min(trailing_edges[n] + b, cols - 1)
            segment = Arc[m, s1b:s2b+1]
            x = np.arange(s1b, s2b + 1)

            #trailing_edge_profiles[m, n] = x[np.argmax(segment)]

            
            
            if len(segment) > 1:
                A = np.vstack([x, np.ones(len(segment))]).T
                bb = np.linalg.lstsq(A, segment, rcond=None)[0]  # Solve Ax = b
                slope, intercept = bb  # Unpack slope and intercept
                p = (threshold - intercept) / slope  # Compute refined edge position
                trailing_edge_profiles[m, n] = p
            

    return leading_edge_profiles, trailing_edge_profiles
'''

'''
#good
def edge_detect_lin(leading_edges, trailing_edges, Arc, b):
    rows, cols = Arc.shape
    leading_edge_profiles = np.zeros((rows, len(leading_edges)))
    trailing_edge_profiles = np.zeros((rows, len(trailing_edges)))
    
    for n in range(len(leading_edges)):
        for m in range(rows):
            # -- Leading edge --
            s1 = max(leading_edges[n] - b, 0)
            s2 = min(leading_edges[n] + b, cols - 1)
            segment = Arc[m, s1:s2+1]
            x = np.arange(s1, s2 + 1)
            if len(segment) > 2:
                # Adaptive threshold at 50% local contrast
                min_val, max_val = np.min(segment), np.max(segment)
                local_thresh = min_val + 0.5 * (max_val - min_val)

                # Use only rising part of signal for fitting (optional)
                above_thresh = np.where(segment >= local_thresh)[0]
                if len(above_thresh) >= 2:
                    idxs = above_thresh[:2]  # Take first two above threshold for fit
                    A = np.vstack([x[idxs], np.ones(len(idxs))]).T
                    bb = np.linalg.lstsq(A, segment[idxs], rcond=None)[0]
                    slope, intercept = bb
                    if abs(slope) > 1e-6:
                        p = (local_thresh - intercept) / slope
                        leading_edge_profiles[m, n] = np.clip(p, s1, s2)
                    else:
                        leading_edge_profiles[m, n] = x[np.argmax(segment)]
                else:
                    leading_edge_profiles[m, n] = x[np.argmax(segment)]
            else:
                leading_edge_profiles[m, n] = x[np.argmax(segment)]

            # -- Trailing edge --
            s1b = max(trailing_edges[n] - b, 0)
            s2b = min(trailing_edges[n] + b, cols - 1)
            segment_b = Arc[m, s1b:s2b+1]
            x_b = np.arange(s1b, s2b + 1)
            if len(segment_b) > 2:
                min_val, max_val = np.min(segment_b), np.max(segment_b)
                local_thresh = min_val + 0.5 * (max_val - min_val)
                above_thresh = np.where(segment_b >= local_thresh)[0]
                if len(above_thresh) >= 2:
                    idxs = above_thresh[-2:]  # Take last two above threshold
                    A = np.vstack([x_b[idxs], np.ones(len(idxs))]).T
                    bb = np.linalg.lstsq(A, segment_b[idxs], rcond=None)[0]
                    slope, intercept = bb
                    if abs(slope) > 1e-6:
                        p = (local_thresh - intercept) / slope
                        trailing_edge_profiles[m, n] = np.clip(p, s1b, s2b)
                    else:
                        trailing_edge_profiles[m, n] = x_b[np.argmax(segment_b)]
                else:
                    trailing_edge_profiles[m, n] = x_b[np.argmax(segment_b)]
            else:
                trailing_edge_profiles[m, n] = x_b[np.argmax(segment_b)]

    return leading_edge_profiles, trailing_edge_profiles

'''

'''
#good
def edge_detect_lin(leading_edges, trailing_edges, Arc, b, threshold):
    rows, cols = Arc.shape
    leading_edge_profiles = np.zeros((rows, len(leading_edges)))
    trailing_edge_profiles = np.zeros((rows, len(trailing_edges)))
    
    for n in range(len(leading_edges)):
        for m in range(rows):
            # --- Leading edge (rising transition) ---
            s1 = max(leading_edges[n] - b, 0)
            s2 = min(leading_edges[n] + b, cols - 1)
            segment = Arc[m, s1:s2+1].astype(np.float32)
            x = np.arange(s1, s2+1, dtype=np.float32)
            pos = None
            # Find first crossing: where intensity goes from below threshold to >= threshold
            for i in range(1, len(segment)):
                if segment[i-1] < threshold and segment[i] >= threshold:
                    # Linear interpolation:
                    pos = x[i-1] + (threshold - segment[i-1]) / (segment[i] - segment[i-1])
                    break
            if pos is None:
                # Fallback: use the maximum intensity position
                pos = x[np.argmax(segment)]
            leading_edge_profiles[m, n] = pos

            # --- Trailing edge (falling transition) ---
            s1b = max(trailing_edges[n] - b, 0)
            s2b = min(trailing_edges[n] + b, cols - 1)
            segment_b = Arc[m, s1b:s2b+1].astype(np.float32)
            x_b = np.arange(s1b, s2b+1, dtype=np.float32)
            pos_b = None
            # Find last crossing: where intensity goes from >= threshold to < threshold
            for i in range(len(segment_b)-1, 0, -1):
                if segment_b[i] < threshold and segment_b[i-1] >= threshold:
                    pos_b = x_b[i-1] + (threshold - segment_b[i-1]) / (segment_b[i] - segment_b[i-1])
                    break
            if pos_b is None:
                pos_b = x_b[np.argmax(segment_b)]
            trailing_edge_profiles[m, n] = pos_b

    return leading_edge_profiles, trailing_edge_profiles
'''

In [ ]:
'''
    row_indices = np.arange(trailingEdgeProfiles.shape[0])
    y_trailing = np.repeat(row_indices, trailingEdgeProfiles.shape[1])
    y_leading  = np.repeat(row_indices, leadingEdgeProfiles.shape[1])
    x_trailing = trailingEdgeProfiles.flatten()
    x_leading  = leadingEdgeProfiles.flatten()

    plt.figure(figsize=(12, 12))
    plt.imshow(image, cmap='gray')
    plt.scatter(x_trailing, y_trailing,s=0.5, c='blue', label='Trailing Edges', marker='o')
    plt.scatter(x_leading, y_leading,s=0.5, c='red', label='Leading Edges', marker='o')
    plt.title("SEM Image with Highlighted Edge Positions")
    plt.xlabel("Pixel Column")
    plt.ylabel("Pixel Row")
    plt.legend()
    plt.tight_layout()
    plt.axis('off')
    plt.show()
'''

In [ ]:
'''#best
def edge_detect_lin(leading_edges, trailing_edges, Arc, b):
    rows, cols = Arc.shape
    leading_edge_profiles = np.zeros((rows, len(leading_edges)))
    trailing_edge_profiles = np.zeros((rows, len(trailing_edges)))
    
    for n in range(len(leading_edges)):
        for m in range(rows):
            # --- Leading edge ---
            s1 = max(leading_edges[n] - b, 0)
            s2 = min(leading_edges[n] + b, cols - 1)
            segment = Arc[m, s1:s2+1].astype(np.float32)
            x = np.arange(s1, s2+1, dtype=np.float32)
            if len(segment) >= 3:
                local_min = segment.min()
                local_max = segment.max()
                mid_val = local_min + 0.5 * (local_max - local_min)
                # Fit quadratic: f(x) = ax^2 + bx + c
                coeffs = np.polyfit(x, segment, 2)
                # Solve a*x^2 + b*x + (c - mid_val) = 0
                a = coeffs[0]
                b_coef = coeffs[1]
                c_val = coeffs[2] - mid_val
                discriminant = b_coef**2 - 4 * a * c_val
                candidate = None
                if discriminant >= 0:
                    sol1 = (-b_coef + np.sqrt(discriminant)) / (2 * a)
                    sol2 = (-b_coef - np.sqrt(discriminant)) / (2 * a)
                    # Choose the solution that falls within the window
                    if s1 <= sol1 <= s2:
                        candidate = sol1
                    elif s1 <= sol2 <= s2:
                        candidate = sol2
                if candidate is None:
                    # Fallback: use the max intensity position
                    candidate = x[np.argmax(segment)]
                leading_edge_profiles[m, n] = candidate
            else:
                leading_edge_profiles[m, n] = x[np.argmax(segment)]
            
            # --- Trailing edge ---
            s1b = max(trailing_edges[n] - b, 0)
            s2b = min(trailing_edges[n] + b, cols - 1)
            segment_b = Arc[m, s1b:s2b+1].astype(np.float32)
            x_b = np.arange(s1b, s2b+1, dtype=np.float32)
            if len(segment_b) >= 3:
                local_min = segment_b.min()
                local_max = segment_b.max()
                mid_val = local_min + 0.5 * (local_max - local_min)
                coeffs = np.polyfit(x_b, segment_b, 2)
                a = coeffs[0]
                b_coef = coeffs[1]
                c_val = coeffs[2] - mid_val
                discriminant = b_coef**2 - 4 * a * c_val
                candidate = None
                if discriminant >= 0:
                    sol1 = (-b_coef + np.sqrt(discriminant)) / (2 * a)
                    sol2 = (-b_coef - np.sqrt(discriminant)) / (2 * a)
                    if s1b <= sol1 <= s2b:
                        candidate = sol1
                    elif s1b <= sol2 <= s2b:
                        candidate = sol2
                if candidate is None:
                    candidate = x_b[np.argmax(segment_b)]
                trailing_edge_profiles[m, n] = candidate
            else:
                trailing_edge_profiles[m, n] = x_b[np.argmax(segment_b)]
    
    return leading_edge_profiles, trailing_edge_profiles

    
    def edge_detect_lin(leading_edges, trailing_edges, Arc, b):
        rows, cols = Arc.shape
        leading_edge_profiles = np.zeros((rows, len(leading_edges)))
        trailing_edge_profiles = np.zeros((rows, len(trailing_edges)))

        for n in range(len(leading_edges)):
            for m in range(rows):
                # --- Leading edge ---
                s1 = max(leading_edges[n] - b, 0)
                s2 = min(leading_edges[n] + b, cols - 1)
                segment = Arc[m, s1:s2+1].astype(np.float32)
                x = np.arange(s1, s2+1, dtype=np.float32)

                if len(segment) >= 7:  # Need at least 7 points for degree 6
                    local_min = segment.min()
                    local_max = segment.max()
                    mid_val = local_min + 0.5 * (local_max - local_min)

                    coeffs = np.polyfit(x, segment, 6)
                    coeffs[-1] -= mid_val  # Subtract mid-level for root finding

                    roots = np.roots(coeffs)
                    real_roots = [r.real for r in roots if np.isreal(r) and s1 <= r.real <= s2]

                    if real_roots:
                        candidate = min(real_roots, key=lambda r: abs(r - (s1 + s2) / 2))
                    else:
                        candidate = x[np.argmax(segment)]
                    leading_edge_profiles[m, n] = candidate
                else:
                    leading_edge_profiles[m, n] = x[np.argmax(segment)]

                # --- Trailing edge ---
                s1b = max(trailing_edges[n] - b, 0)
                s2b = min(trailing_edges[n] + b, cols - 1)
                segment_b = Arc[m, s1b:s2b+1].astype(np.float32)
                x_b = np.arange(s1b, s2b+1, dtype=np.float32)

                if len(segment_b) >= 7:
                    local_min = segment_b.min()
                    local_max = segment_b.max()
                    mid_val = local_min + 0.5 * (local_max - local_min)

                    coeffs_b = np.polyfit(x_b, segment_b, 6)
                    coeffs_b[-1] -= mid_val

                    roots_b = np.roots(coeffs_b)
                    real_roots_b = [r.real for r in roots_b if np.isreal(r) and s1b <= r.real <= s2b]

                    if real_roots_b:
                        candidate_b = min(real_roots_b, key=lambda r: abs(r - (s1b + s2b) / 2))
                    else:
                        candidate_b = x_b[np.argmax(segment_b)]
                    trailing_edge_profiles[m, n] = candidate_b
                else:
                    trailing_edge_profiles[m, n] = x_b[np.argmax(segment_b)]

        return leading_edge_profiles, trailing_edge_profiles

'''


### PSD Curve

In [ ]:
'''
def hhcf(x, ps):
    N = x.shape[0]
    M = round(N / 4)
    G = np.zeros(M)
    for m in range(1, M + 1):
        for n in range(m, N - m):
            G[m - 1] += (x[n - m] - x[n + m]) ** 2
        G[m - 1] /= (N - 2 * m)
    r = ps * (2 * np.arange(1, M + 1) + 1)
    return G, r

def hhcfmod(r, a, b, c, d):  
    return a * np.exp(-r/b) + c + d

def Palasantzas2(beta, freqx):
    return beta[0] * np.exp(-beta[1] * freqx) + beta[2] + beta[3] * freqx**2

def objective_function(beta, freqx, mF_data):
    return np.sum((Palasantzas2(beta, freqx) - mF_data) ** 2)
'''

In [ ]:
'''def fit_PSD(freqx, mF_data, CF, Alpha, FN, LN, MI=10000):
    Lx = len(mF_data)
    low_band = mF_data[:FN]
    high_band = mF_data[-LN:]
    A0 = np.sqrt(max(np.nanmean(low_band) - np.nanmean(high_band), 1e-6))
    C0 = np.nanmean(high_band)
    beta0 = [A0, CF, C0, Alpha]
    
    result = minimize(objective_function, beta0, args=(freqx, mF_data), options={'maxiter': MI})
    betaf = result.x
    mF_fit = Palasantzas2(betaf, freqx)
    betan = betaf.copy()
    betan[2] = 0
    mF_fit_unbiased = Palasantzas2(betan, freqx)
    return betaf, mF_fit, mF_fit_unbiased

def analyze_psd(leadingEdgeProfiles, trailingEdgeProfiles, pixelWidthNm):
    LW = leadingEdgeProfiles - trailingEdgeProfiles
    
    LWR_edge = 3 * np.mean(np.std(LW, axis=0))
    LER_leading = 3 * np.mean(np.std(leadingEdgeProfiles, axis=0))
    LER_trailing = 3 * np.mean(np.std(trailingEdgeProfiles, axis=0))
    LER_edge = (LER_leading + LER_trailing) / 2
    
    print(f"LWR (3σ, edge-based): {LWR_edge:.3f} nm")
    print(f"LER (3σ, edge-based): {LER_edge:.3f} nm")
    
    N = LW.shape[0]
    Fs = 1 / pixelWidthNm
    LW_fft = np.fft.rfft(LW, axis=0)
    mF_LW = np.nanmean(np.abs(LW_fft)**2, axis=1) / (Fs * N)
    
    freq = np.fft.rfftfreq(N, d=pixelWidthNm)
    delta_f = freq[1] - freq[0] if len(freq) > 1 else 1
    
    FN = 5
    LN = 50
    CF = 0.02
    Alpha = 2
    Lx = len(mF_LW)
    low_band = mF_LW[:FN]
    high_band = mF_LW[-LN:]
    A0 = np.sqrt(max(np.nanmean(low_band) - np.nanmean(high_band), 1e-6))
    C0 = np.nanmean(high_band)
    beta0 = [A0, CF, C0, Alpha]
    
    result = minimize(objective_function, beta0, args=(freq, mF_LW), options={'maxiter':10000})
    betaf_LW = result.x
    print("Fitted PSD parameters for LWR (biased):", betaf_LW)
    mF_fit = Palasantzas2(betaf_LW, freq)
    
    betan = betaf_LW.copy()
    betan[2] = 0
    mF_fit_unbiased = Palasantzas2(betan, freq)
    
    sigma2_LWR_biased = np.trapz(mF_LW, freq)
    sigma2_LWR_unbiased = np.trapz(np.clip(mF_LW - betaf_LW[2] / (Fs * N), 0, None), freq)
    
    LWR_psd_biased = 3 * np.sqrt(sigma2_LWR_biased)
    LWR_psd_unbiased = 3 * np.sqrt(sigma2_LWR_unbiased)
    print(f"LWR (3σ, PSD biased): {LWR_psd_biased:.3f} nm")
    print(f"LWR (3σ, PSD unbiased): {LWR_psd_unbiased:.3f} nm")
    
    LE_fft = np.fft.rfft(leadingEdgeProfiles, axis=0)
    TE_fft = np.fft.rfft(trailingEdgeProfiles, axis=0)
    mF_LE = np.nanmean(np.abs(LE_fft)**2, axis=1) / (Fs * N)
    mF_TE = np.nanmean(np.abs(TE_fft)**2, axis=1) / (Fs * N)
    mF_LER = (mF_LE + mF_TE) / 2
    
    low_band_LE = mF_LER[:FN]
    high_band_LE = mF_LER[-LN:]
    A0_LE = np.sqrt(max(np.nanmean(low_band_LE) - np.nanmean(high_band_LE), 1e-6))
    C0_LE = np.nanmean(high_band_LE)
    beta0_LE = [A0_LE, CF, C0_LE, Alpha]
    
    result_LE = minimize(objective_function, beta0_LE, args=(freq, mF_LER), options={'maxiter':10000})
    betaf_LER = result_LE.x
    print("Fitted PSD parameters for LER (biased):", betaf_LER)
    mF_LE_fit = Palasantzas2(betaf_LER, freq)
    betan_LE = betaf_LER.copy()
    betan_LE[2] = 0
    mF_LE_fit_unbiased = Palasantzas2(betan_LE, freq)
    
    sigma2_LER_biased = np.trapz(mF_LER, freq)
    sigma2_LER_unbiased = np.trapz(np.clip(mF_LER - betaf_LER[2] / (Fs * N), 0, None), freq)
    
    LER_psd_biased = 3 * np.sqrt(sigma2_LER_biased)
    LER_psd_unbiased = 3 * np.sqrt(sigma2_LER_unbiased)
    print(f"LER (3σ, PSD biased): {LER_psd_biased:.3f} nm")
    print(f"LER (3σ, PSD unbiased): {LER_psd_unbiased:.3f} nm")
    
    plt.figure(figsize=(10, 6))
    plt.plot(freq, mF_LW, 'b', label='LWR Measured PSD (biased)')
    #plt.plot(freq, np.clip(mF_LW - betaf_LW[2] / (Fs * N), 0, None), 'r--', label='LWR Measured PSD (unbiased)')
    #plt.plot(freq, mF_fit / (Fs * N), 'g:', label='LWR PSD Fit (biased)')
    #plt.plot(freq, mF_fit_unbiased / (Fs * N), 'm-.', label='LWR PSD Fit (unbiased)')
    plt.xlabel('Frequency (1/nm)')
    plt.ylabel('PSD (nm^3)')
    plt.xscale('log')
    plt.yscale('log')
    plt.title('PSD of LWR')
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 6))
    plt.plot(freq, mF_LER, 'b', label='LER Measured PSD (biased)')
    #plt.plot(freq, np.clip(mF_LER - betaf_LER[2] / (Fs * N), 0, None), 'r--', label='LER Measured PSD (unbiased)')
    #plt.plot(freq, mF_LE_fit / (Fs * N), 'g:', label='LER PSD Fit (biased)')
    #plt.plot(freq, mF_LE_fit_unbiased / (Fs * N), 'm-.', label='LER PSD Fit (unbiased)')
    plt.xlabel('Frequency (1/nm)')
    plt.ylabel('PSD (nm^3)')
    plt.xscale('log')
    plt.yscale('log')
    plt.title('PSD of LER')
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    results = {
        "LWR_edge_based (3σ)": LWR_edge,
        "LER_edge_based (3σ)": LER_edge,
        "LWR_PSD_biased (3σ)": LWR_psd_biased,
        "LWR_PSD_unbiased (3σ)": LWR_psd_unbiased,
        "LER_PSD_biased (3σ)": LER_psd_biased,
        "LER_PSD_unbiased (3σ)": LER_psd_unbiased,
        "Fitted PSD parameters LWR": betaf_LW,
        "Fitted PSD parameters LER": betaf_LER
    }
    return results'''

In [ ]:
'''import numpy as np
from scipy.optimize import minimize
from scipy.signal.windows import dpss
import matplotlib.pyplot as plt

def calculate_ler_lwr(leading_edge_profiles, trailing_edge_profiles, pixel_width_nm,
                      NW=4, K=6, freq_low_cutoff=0.1, freq_high_cutoff=0.9,
                      plot_results=False):
    # Preprocess edge profiles
    LW = leading_edge_profiles - trailing_edge_profiles
    combined_edges = np.concatenate([leading_edge_profiles, trailing_edge_profiles], axis=1)
    
    # Remove profile means
    LW_centered = LW - LW.mean(axis=0)
    edges_centered = combined_edges - combined_edges.mean(axis=0)

    # Define multitaper PSD function
    def multitaper_psd(signal):
        N = len(signal)
        dpss_result = dpss(N, NW, Kmax=K)  # Capture all outputs
        tapers = dpss_result[0] if isinstance(dpss_result, tuple) else dpss_result
        
        # Ensure proper dimensions
        if tapers.ndim == 1:
            tapers = tapers.reshape(-1, 1)
            
        num_tapers = min(K, tapers.shape[1])  # Use available tapers
        psd_mt = np.zeros(N)
        
        for k in range(num_tapers):
            tapered = tapers[:, k] * signal
            psd = np.abs(np.fft.fft(tapered))**2
            psd_mt += psd
            
        psd_mt /= num_tapers
        freq = np.fft.fftfreq(N, d=pixel_width_nm)[:N//2]
        return freq, psd_mt[:N//2]


    # Define PSD model and objective function
    def Palasantzas_model(freq, sigma, xi, alpha, Nl):
        return (xi * sigma**2) / (1 + (xi*freq)**2)**(0.5 + alpha) + Nl

    def objective(params, freq, psd):
        sigma, xi, alpha, Nl = params
        return np.sum((Palasantzas_model(freq, sigma, xi, alpha, Nl) - psd)**2)

    # Initialize parameters using frequency ranges
    def initialize_params(freq, psd):
        mask_low = freq < freq_low_cutoff
        mask_high = freq > freq_high_cutoff
        
        sigma_init = np.sqrt(np.mean(psd[mask_low]))
        xi_init = 1/(2*np.pi*np.median(freq[mask_low]))
        Nl_init = np.mean(psd[mask_high])
        
        return [sigma_init, xi_init, 0.5, Nl_init]

    # Process LW profiles (LWR calculation)
    freq_lw, psd_lw = multitaper_psd(LW_centered[:, 0])
    params_init = initialize_params(freq_lw, psd_lw)
    res = minimize(objective, params_init, args=(freq_lw, psd_lw))
    sigma, xi, alpha, Nl = res.x
    
    # Calculate biased/unbiased LWR
    psd_lw_unbiased = psd_lw - Nl
    psd_lw_unbiased[psd_lw_unbiased < 0] = 0
    LWR_biased = 3*np.sqrt(np.trapz(psd_lw, freq_lw))
    LWR_unbiased = 3*np.sqrt(np.trapz(psd_lw_unbiased, freq_lw))

    # Process combined edges (LER calculation)
    freq_ler, psd_ler = multitaper_psd(edges_centered[:, 0])
    params_init_ler = initialize_params(freq_ler, psd_ler)
    res_ler = minimize(objective, params_init_ler, args=(freq_ler, psd_ler))
    sigma_ler, xi_ler, alpha_ler, Nl_ler = res_ler.x
    
    # Calculate biased/unbiased LER
    psd_ler_unbiased = psd_ler - Nl_ler
    psd_ler_unbiased[psd_ler_unbiased < 0] = 0
    LER_biased = 3*np.sqrt(np.trapz(psd_ler, freq_ler))
    LER_unbiased = 3*np.sqrt(np.trapz(psd_ler_unbiased, freq_ler))

    # Generate plots if requested
    if plot_results:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
        
        # LWR plot
        ax1.loglog(freq_lw, psd_lw, label='Biased PSD')
        ax1.loglog(freq_lw, psd_lw_unbiased, label='Unbiased PSD')
        ax1.loglog(freq_lw, Palasantzas_model(freq_lw, sigma, xi, alpha, 0), 
                 '--', label='Model Fit')
        ax1.set_title('LWR Power Spectral Density')
        ax1.set_ylabel('PSD (nm³)')
        ax1.legend()
        
        # LER plot
        ax2.loglog(freq_ler, psd_ler, label='Biased PSD')
        ax2.loglog(freq_ler, psd_ler_unbiased, label='Unbiased PSD')
        ax2.loglog(freq_ler, Palasantzas_model(freq_ler, sigma_ler, xi_ler, alpha_ler, 0),
                 '--', label='Model Fit')
        ax2.set_title('LER Power Spectral Density')
        ax2.set_xlabel('Frequency (1/nm)')
        ax2.set_ylabel('PSD (nm³)')
        ax2.legend()
        
        plt.tight_layout()
        plt.show()

    return {
        'LWR_biased_3sigma': LWR_biased,
        'LWR_unbiased_3sigma': LWR_unbiased,
        'LER_biased_3sigma': LER_biased,
        'LER_unbiased_3sigma': LER_unbiased,
        'PSD_parameters_LWR': {'sigma': sigma, 'xi': xi, 'alpha': alpha, 'Nl': Nl},
        'PSD_parameters_LER': {'sigma': sigma_ler, 'xi': xi_ler, 
                              'alpha': alpha_ler, 'Nl': Nl_ler},
        'frequency_LWR': freq_lw,
        'PSD_LWR': psd_lw,
        'PSD_LWR_unbiased': psd_lw_unbiased,
        'frequency_LER': freq_ler,
        'PSD_LER': psd_ler,
        'PSD_LER_unbiased': psd_ler_unbiased
    }'''

In [ ]:
'''
import numpy as np
from scipy.signal.windows import dpss
from scipy.optimize import minimize
import matplotlib.pyplot as plt

def palasantzas_psd(freq, params):
    """Palasantzas model: PSD(f) = ξ·σ² / (1 + (ξ·f)²)**(0.5 + α) + Nl"""
    σ, ξ, α, Nl = params
    return (ξ * σ**2) / (1 + (ξ * freq)**2)**(0.5 + α) + Nl

def objective(params, freq, psd):
    """Least‐squares objective for PSD fit."""
    model = palasantzas_psd(freq, params)
    return np.sum((psd - model)**2)

def multitaper_psd(width_profiles, dx, NW, K):
    """
    Multitaper PSD estimate.
      width_profiles: array shape (N, M)
      dx: sampling step (nm)
      NW: time–bandwidth product
      K: number of tapers
    Returns (freq, psd_mt)
    """
    N, M = width_profiles.shape
    tapers, _ = dpss(N, NW, Kmax=K, return_ratios=True)
    acc = np.zeros(N//2+1)
    for m in range(M):
        x = width_profiles[:, m]
        for k in range(K):
            X = np.fft.rfft(x * tapers[k])
            acc += np.abs(X)**2
    psd_mt = (dx**2 / (N*M*K)) * acc
    freq = np.fft.rfftfreq(N, d=dx)
    return freq, psd_mt

def compute_ler_lwr(freq, psd_mt, fit_alpha, fit_ξ):
    """
    Fit Palasantzas model to psd_mt, then compute
    biased & unbiased LER/LWR (3σ).
    fit_alpha, fit_ξ: initial guesses for α, ξ
    """
    # Estimate noise floor Nl from high‑freq tail:
    Nl0 = np.mean(psd_mt[-10:])
    # Estimate σ from low‑freq minus Nl0:
    σ0 = np.sqrt(max(np.mean(psd_mt[:5]) - Nl0, 1e-12))
    p0 = [σ0, fit_ξ, fit_alpha, Nl0]

    res = minimize(objective, p0, args=(freq, psd_mt), method='Nelder-Mead')
    σ, ξ, α, Nl = res.x

    # Biased roughness:
    var_biased = np.trapz(psd_mt, freq)
    rough_biased = 3 * np.sqrt(var_biased)

    # Unbiased roughness (subtract white noise floor):
    psd_unb = np.clip(psd_mt - Nl, 0, None)
    var_unb = np.trapz(psd_unb, freq)
    rough_unb = 3 * np.sqrt(var_unb)

    return {
        'σ': σ, 'ξ': ξ, 'α': α, 'Nl': Nl,
        'rough_biased': rough_biased,
        'rough_unbiased': rough_unb,
        'freq': freq,
        'psd_mt': psd_mt,
        'psd_fit': palasantzas_psd(freq, res.x),
        'psd_unbiased': psd_unb
    }
'''

### Adding noise to image

In [9]:
'''
import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as iio

def add_noise_to_image(image, mode='white', **kwargs):
    orig_dtype = image.dtype
    is_int     = np.issubdtype(orig_dtype, np.integer)

    img = image.astype(np.float32)
    if is_int:
        FS = np.iinfo(orig_dtype).max
        img /= FS
    else:
        img = np.clip(img, 0., 1.)

    noisy = img.copy()
    H, W = img.shape

    if mode == 'white':
        amp   = kwargs.get('amplitude', 0.05)
        noise = np.random.uniform(-amp, amp, size=img.shape)
        noisy += noise

    elif mode == 'gaussian':
        sigma = kwargs.get('sigma', 0.05)
        noise = np.random.normal(0, sigma, size=img.shape)
        noisy += noise

    elif mode == 'poisson':
        scale = kwargs.get('scale', 50)
        vals  = np.random.poisson(img * scale) / float(scale)
        noisy = vals

    elif mode == 'salt_pepper':
        amount    = kwargs.get('amount', 0.05)
        s_vs_p    = kwargs.get('salt_vs_pepper', 0.5)
        noisy     = img.copy()

        num_salt  = int(np.ceil(amount * H * W * s_vs_p))
        coords    = (np.random.randint(0, H, num_salt),
                     np.random.randint(0, W, num_salt))
        noisy[coords] = 1.0

        num_pepper = int(np.ceil(amount * H * W * (1 - s_vs_p)))
        coords     = (np.random.randint(0, H, num_pepper),
                      np.random.randint(0, W, num_pepper))
        noisy[coords] = 0.0

    else:
        raise ValueError("mode must be 'white','gaussian','poisson' or 'salt_pepper'")

    noisy = np.clip(noisy, 0., 1.)
    if is_int:
        noisy = (noisy * FS).round().astype(orig_dtype)
    return noisy

if __name__ == "__main__":
    img = iio.imread("../Images/80 nm_box 4_8 sec_024_dose_15.tif")   # e.g. uint16
    noisy = add_noise_to_image(img, mode='white', amplitude=0.1)

    fig, ax = plt.subplots(1,2, figsize=(8,4))
    vmin, vmax = img.min(), img.max()
    ax[0].imshow(img,   cmap='gray', vmin=vmin, vmax=vmax); ax[0].set_title("Orig"); ax[0].axis('off')
    ax[1].imshow(noisy, cmap='gray', vmin=vmin, vmax=vmax); ax[1].set_title("White noise"); ax[1].axis('off')
    plt.tight_layout()
    plt.show()

    iio.imwrite("noisy_white.tif", noisy)

'''

'\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport imageio.v3 as iio\n\ndef add_noise_to_image(image, mode=\'white\', **kwargs):\n    orig_dtype = image.dtype\n    is_int     = np.issubdtype(orig_dtype, np.integer)\n\n    img = image.astype(np.float32)\n    if is_int:\n        FS = np.iinfo(orig_dtype).max\n        img /= FS\n    else:\n        img = np.clip(img, 0., 1.)\n\n    noisy = img.copy()\n    H, W = img.shape\n\n    if mode == \'white\':\n        amp   = kwargs.get(\'amplitude\', 0.05)\n        noise = np.random.uniform(-amp, amp, size=img.shape)\n        noisy += noise\n\n    elif mode == \'gaussian\':\n        sigma = kwargs.get(\'sigma\', 0.05)\n        noise = np.random.normal(0, sigma, size=img.shape)\n        noisy += noise\n\n    elif mode == \'poisson\':\n        scale = kwargs.get(\'scale\', 50)\n        vals  = np.random.poisson(img * scale) / float(scale)\n        noisy = vals\n\n    elif mode == \'salt_pepper\':\n        amount    = kwargs.get(\'amount\

In [10]:
'''
import matplotlib.pyplot as plt
import imageio.v3 as iio

def add_noise_to_image(image, mode='white', **kwargs):
    orig_dtype = image.dtype
    is_int     = np.issubdtype(orig_dtype, np.integer)

    img = image.astype(np.float32)
    if is_int:
        FS = np.iinfo(orig_dtype).max
        img /= FS
    else:
        img = np.clip(img, 0., 1.)

    noisy = img.copy()
    H, W = img.shape

    if mode == 'white':
        amp   = kwargs.get('amplitude', 0.05)
        noise = np.random.uniform(-amp, amp, size=img.shape)
        noisy += noise

    elif mode == 'gaussian':
        sigma = kwargs.get('sigma', 0.05)
        noise = np.random.normal(0, sigma, size=img.shape)
        noisy += noise

    elif mode == 'poisson':
        scale = kwargs.get('scale', 50)
        vals  = np.random.poisson(img * scale) / float(scale)
        noisy = vals

    elif mode == 'salt_pepper':
        amount    = kwargs.get('amount', 0.05)
        s_vs_p    = kwargs.get('salt_vs_pepper', 0.5)
        noisy     = img.copy()

        num_salt  = int(np.ceil(amount * H * W * s_vs_p))
        coords    = (np.random.randint(0, H, num_salt),
                     np.random.randint(0, W, num_salt))
        noisy[coords] = 1.0

        num_pepper = int(np.ceil(amount * H * W * (1 - s_vs_p)))
        coords     = (np.random.randint(0, H, num_pepper),
                      np.random.randint(0, W, num_pepper))
        noisy[coords] = 0.0

    else:
        raise ValueError("mode must be 'white','gaussian','poisson' or 'salt_pepper'")

    noisy = np.clip(noisy, 0., 1.)
    if is_int:
        noisy = (noisy * FS).round().astype(orig_dtype)
    return noisy

if __name__ == "__main__":
    img = iio.imread("../Images/80 nm_box 4_8 sec_024_dose_15.tif")   # e.g. uint16
    noisy = add_noise_to_image(img, mode='white', amplitude=0.1)

    fig, ax = plt.subplots(1,2, figsize=(8,4))
    vmin, vmax = img.min(), img.max()
    ax[0].imshow(img,   cmap='gray', vmin=vmin, vmax=vmax); ax[0].set_title("Orig"); ax[0].axis('off')
    ax[1].imshow(noisy, cmap='gray', vmin=vmin, vmax=vmax); ax[1].set_title("White noise"); ax[1].axis('off')
    plt.tight_layout()
    plt.show()

    iio.imwrite("noisy_white.tif", noisy)

'''

'\nimport matplotlib.pyplot as plt\nimport imageio.v3 as iio\n\ndef add_noise_to_image(image, mode=\'white\', **kwargs):\n    orig_dtype = image.dtype\n    is_int     = np.issubdtype(orig_dtype, np.integer)\n\n    img = image.astype(np.float32)\n    if is_int:\n        FS = np.iinfo(orig_dtype).max\n        img /= FS\n    else:\n        img = np.clip(img, 0., 1.)\n\n    noisy = img.copy()\n    H, W = img.shape\n\n    if mode == \'white\':\n        amp   = kwargs.get(\'amplitude\', 0.05)\n        noise = np.random.uniform(-amp, amp, size=img.shape)\n        noisy += noise\n\n    elif mode == \'gaussian\':\n        sigma = kwargs.get(\'sigma\', 0.05)\n        noise = np.random.normal(0, sigma, size=img.shape)\n        noisy += noise\n\n    elif mode == \'poisson\':\n        scale = kwargs.get(\'scale\', 50)\n        vals  = np.random.poisson(img * scale) / float(scale)\n        noisy = vals\n\n    elif mode == \'salt_pepper\':\n        amount    = kwargs.get(\'amount\', 0.05)\n        s_